## 1. Setup


In [ ]:
from google.colab import drive
from pathlib import Path
import sys, shutil, subprocess, importlib.metadata, os

drive.mount('/content/drive')
ROOT = Path(os.environ.get('A3_ROOT', '/content/drive/MyDrive/KSE_AI301_A3'))
WORK = Path(os.environ.get('A3_WORK', '/content/a3'))
PROJECT = WORK / 'project'
PROJECT.mkdir(parents=True, exist_ok=True)
for name in ['part2.py', 'part2_data.py', 'part2_split.json', 'requirements-part2.txt',
             'ner_data.py', 'pilot_ner.py', 'recurrent_model.py', 'metrics.py', 'validation.py']:
    shutil.copy2(ROOT / 'project' / name, PROJECT / name)
sys.path.insert(0, str(PROJECT))
os.environ['MPLCONFIGDIR'] = '/tmp/a3_matplotlib'
import torch
version = importlib.metadata.version('torch')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements-part2.txt')])
assert importlib.metadata.version('torch') == version
from part2_data import file_hash, load_data, prepare_data
from part2 import CONFIG, gpu, train, load_model, evaluate, cross_validate, predict
ACTION = 'predict'  # 'train' starts a separate reproduction from scratch.
assert ACTION in {'predict', 'train'}
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if ACTION == 'train':
    print(gpu())
print('Device:', DEVICE, 'Action:', ACTION)


## 2. Load data and frozen split


In [ ]:
import json
spec = json.loads((PROJECT / 'part2_split.json').read_text())
DATA = WORK / 'data'
DATA.mkdir(parents=True, exist_ok=True)
for name, expected in spec['files'].items():
    destination = DATA / name
    if not destination.exists() or file_hash(destination) != expected:
        shutil.copy2(ROOT / 'data' / name, destination)
frames, spec = load_data(DATA, PROJECT / 'part2_split.json')


## 3. Prepare data


In [ ]:
data = prepare_data(frames, spec)
print({name: len(docs) for name, docs in data.items()})
RESULTS = ROOT / 'results' / 'part2'


## 4. Train selected BiLSTM


In [ ]:
if ACTION == 'train':
    result = train(CONFIG, data['fit'], data['checkpoint'], spec, RESULTS / 'model', WORK / 'runs')
    checkpoint = Path(result['checkpoint'])
else:
    checkpoint = ROOT / 'part2_model' / 'checkpoint.pt'
model, state = load_model(checkpoint, spec, DEVICE, reference=ACTION == 'predict')


## 5. Evaluate


In [ ]:
metrics = evaluate(model, state, data, spec, RESULTS / ACTION, checkpoint, reference=ACTION == 'predict')
print(metrics)
print('Recorded baseline CV:', spec['reference']['cv_metrics']['f1'])
print('Recorded independent audit:', spec['reference']['metrics']['audit']['f1'])
if ACTION == 'train':
    cross_validate(data, spec, CONFIG, result, RESULTS / 'cv')
    cross_validate(data, spec, CONFIG, result, RESULTS / 'av', domain=True)


## 6. Generate submission.csv


In [ ]:
submission = predict(model, state, data['test'], spec, RESULTS / ACTION / 'submission.csv', reference=ACTION == 'predict')
print(submission)
